## Module 10: Hyperparameter Tuning and Debugging

In [1]:
# Grid Search
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import itertools
import json
from datetime import datetime

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

n = 1500
X = torch.randn(n, 15)
y = (X[:, :3].sum(1) > 0).float().unsqueeze(1)
X_tr, X_val = X[:1200], X[1200:]
y_tr, y_val = y[:1200], y[1200:]

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)

def build_model(hidden_size, dropout_p, n_layers = 2):
    layers = [nn.Linear(15, hidden_size), nn.ReLU()]
    if dropout_p > 0:
        layers.append(nn.Dropout(dropout_p))
    for _ in range(n_layers - 1):
        layers += [nn.Linear(hidden_size, hidden_size), nn.ReLU()]
        if dropout_p > 0:
            layers.append(nn.Dropout(dropout_p))
    layers.append(nn.Linear(hidden_size, 1))
    return nn.Sequential(*layers).to(device)

def quick_train(model, lr, weight_decay, n_epochs = 30):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr = lr, weight_decay = weight_decay)
    best_val = float('inf')

    for epoch in range(n_epochs):
        model.train()
        for bX, by in train_loader:
            bX, by  = bX.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bX), by)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for bX, by in val_loader:
                bX, by = bX.to(device), by.to(device)
                val_loss += criterion(model(bX), by).item() * bX.size(0)
        val_loss /= len(val_loader.dataset)
        best_val = min(best_val, val_loss)

    return best_val

param_grid = {
    'lr'  : [1e-4, 1e-3, 3e-3], 'weight_decay': [1e-4, 1e-3, 1e-2], 'hidden_size': [64, 128, 256]
}

keys = list(param_grid.keys())
values = list(param_grid.values())
grid = list(itertools.product(*values))

print(f"Grid search: {len(grid)} configurations\n")
print(f"{'lr':>8} {'wd':>8} {'hidden':>8} {'val_loss':>10}")

results = []
for combo in grid:
    config = dict(zip(keys, combo))
    model  = build_model(config['hidden_size'], dropout_p=0.0)
    val_loss = quick_train(model, config['lr'], config['weight_decay'], n_epochs=25)

    results.append({**config, 'val_loss': val_loss})
    print(f"{config['lr']:>8.0e} {config['weight_decay']:>8.0e} "
          f"{config['hidden_size']:>8} {val_loss:>10.4f}")

best = min(results, key=lambda x: x['val_loss'])
print(f"\n✅ Best configuration:")
for k, v in best.items():
    print(f"   {k}: {v}")

Grid search: 27 configurations

      lr       wd   hidden   val_loss
   1e-04    1e-04       64     0.3746
   1e-04    1e-04      128     0.2140
   1e-04    1e-04      256     0.1307
   1e-04    1e-03       64     0.3765
   1e-04    1e-03      128     0.2223
   1e-04    1e-03      256     0.1299
   1e-04    1e-02       64     0.4067
   1e-04    1e-02      128     0.2092
   1e-04    1e-02      256     0.1410
   1e-03    1e-04       64     0.0572
   1e-03    1e-04      128     0.0711
   1e-03    1e-04      256     0.0693
   1e-03    1e-03       64     0.0781
   1e-03    1e-03      128     0.0858
   1e-03    1e-03      256     0.0851
   1e-03    1e-02       64     0.0718
   1e-03    1e-02      128     0.0805
   1e-03    1e-02      256     0.0829
   3e-03    1e-04       64     0.0916
   3e-03    1e-04      128     0.0860
   3e-03    1e-04      256     0.0890
   3e-03    1e-03       64     0.0863
   3e-03    1e-03      128     0.0758
   3e-03    1e-03      256     0.0874
   3e-03    1e-02 

In [2]:
# Random Search implementation
import random
import numpy as np

def sample_config(search_space: dict) -> dict:
    config = {}
    for key, spec in search_space.items():
        if spec[0] == 'log_uniform':
            config[key] = 10 ** random.uniform(np.log10(spec[1]), np.log10(spec[2]))
        elif spec[0] == 'uniform':
            config[key] = random.uniform(spec[1], spec[2])
        elif spec[0] == 'choice':
            config[key] = random.choice(spec[1])
        elif spec[0] == 'int':
            config[key] = random.randint(spec[1], spec[2])
    return config

search_space = {
    'lr' : ('log_uniform', 1e-5, 1e-1), 'weight_decay' : ('log_uniform', 1e-5, 1e-1), 
    'hidden_size' : ('choice', [64, 128, 256, 512]), 'dropout_p' : ('uniform', 0.0, 0.5),
    'n_layers' : ('int', 1, 4)}

n_trials = 20
random.seed(42)
print(f'Random Search: {n_trials} trials\n')
print(f"{'lr':>8} {'wd':>8} {'hidden':>8} {'drop':>6} {'layers':>7} {'val_loss':>10}")

rs_results = []
for trial in range(n_trials):
    config = sample_config(search_space)
    model = build_model(config['hidden_size'], config['dropout_p'], config['n_layers'])
    val_loss = quick_train(model, config['lr'], config['weight_decay'], n_epochs = 20)
    config['val_loss'] = val_loss
    rs_results.append(config)

    print(f"{config['lr']:>8.1e} {config['weight_decay']:>8.1e} "
          f"{config['hidden_size']:>8} {config['dropout_p']:>6.2f} "
              f"{config['n_layers']:>7} {val_loss:>10.4f}")

best_rs = min(rs_results, key=lambda x: x['val_loss'])
print(f"\n✅ Best (random search): val_loss={best_rs['val_loss']:.4f}")
for k, v in best_rs.items():
    if k != 'val_loss':
        print(f"   {k}: {v:.4f}" if isinstance(v, float) else f"   {k}: {v}")

Random Search: 20 trials

      lr       wd   hidden   drop  layers   val_loss
 3.6e-03  1.3e-05      256   0.12       2     0.0762
 8.8e-03  5.1e-03       64   0.30       1     0.0432
 1.3e-05  7.5e-05       64   0.28       4     0.6889
 7.6e-05  2.3e-03       64   0.38       2     0.6328
 6.2e-03  2.3e-04      128   0.11       3     0.0929
 2.6e-05  3.3e-04      256   0.42       3     0.5312
 1.7e-02  8.3e-03       64   0.49       4     0.0747
 2.1e-05  1.5e-04      256   0.29       1     0.6387
 1.5e-05  8.2e-05      256   0.49       2     0.6236
 2.9e-02  3.3e-04      512   0.32       3     0.0897
 4.5e-05  2.6e-04      256   0.35       1     0.6122
 2.7e-03  4.8e-05      128   0.08       4     0.1067
 1.2e-04  5.0e-02      128   0.34       1     0.5460
 8.2e-05  1.3e-05      256   0.20       1     0.5245
 7.0e-05  5.9e-02      256   0.11       4     0.1081
 3.8e-04  4.6e-02      512   0.07       2     0.0815
 9.7e-05  1.8e-03      256   0.37       4     0.1122
 3.9e-02  4.0e-04   

In [6]:
# Optuna
# !pip install optuna

import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

optuna.logging.set_verbosity(optuna.logging.WARNING)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

X_tr_t = X_tr.to(device)
y_tr_t = y_tr.to(device)
X_val_t = X_val.to(device)
y_val_t = y_val.to(device)

def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log = True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-1, log = True)
    hidden_size = trial.suggest_categorical('hidden_size', [64, 128, 256, 512])
    n_layers = trial.suggest_int('n_layers', 1, 4)
    dropout_p = trial.suggest_float('dropout_p', 0.0, 0.5)
    activation_name = trial.suggest_categorical('activation_name', ['relu', 'leaky_relu'])
    
    activation = nn.ReLU() if activation_name == 'relu' else nn.LeakyReLU(0.1)
    layers = [nn.Linear(15, hidden_size), activation]
    if dropout_p > 0:
        layers.append(nn.Dropout(dropout_p))
    for _ in range(n_layers - 1):
        layers += [nn.Linear(hidden_size, hidden_size), activation]
        if dropout_p > 0:
            layers.append(nn.Dropout(dropout_p))
    layers.append(nn.Linear(hidden_size, 1))

    model = nn.Sequential(*layers).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr = lr, weight_decay = weight_decay)

    train_loader_opt = DataLoader(TensorDataset(X_tr_t.cpu(), y_tr_t.cpu()), batch_size=64, shuffle=True)
    best_val_loss = float('inf')

    for epoch in range(30):
        model.train()
        for bX, by in train_loader_opt:
            bX, by = bX.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bX), by)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            logits   = model(X_val_t)
            val_loss = criterion(logits, y_val_t).item()

        best_val_loss = min(best_val_loss, val_loss)

        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_loss

study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler(seed = 42),
                           pruner = optuna.pruners.MedianPruner(n_startup_trials = 5, n_warmup_steps = 10))
print("Running Optuna optimization (50 trials)...\n")
study.optimize(objective, n_trials=50, show_progress_bar=False)

print(f"Best trial:")
print(f"  Val loss: {study.best_value:.4f}")
print(f"  Config:")
for k, v in study.best_params.items():
    print(f"    {k}: {v:.4f}" if isinstance(v, float) else f"    {k}: {v}")

try:
    importances = optuna.importance.get_param_importances(study)
    print(f'\nHyperparameter importance (higher = more impact on loss):')
    for name, importance in sorted(importances.items(), key = lambda x: x[1], reverse = True):
        bar = "█" * int(importance * 30)
        print(f"  {name:<20}: {importance:.3f}  {bar}")
except Exception:
    pass

print(f"\nTop 10 trials (sorted by val_loss):")
print(f"{'Trial':>7} {'Val Loss':>10} {'LR':>10} {'WD':>10} {'Hidden':>8}")
print("-" * 50)
completed = [t for t in study.trials
             if t.state == optuna.trial.TrialState.COMPLETE]
for t in sorted(completed, key=lambda x: x.value)[:10]:
    print(f"{t.number:>7} {t.value:>10.4f} "
          f"{t.params.get('lr', 0):>10.2e} "
          f"{t.params.get('weight_decay', 0):>10.2e} "
          f"{t.params.get('hidden_size', 0):>8}")

Running Optuna optimization (50 trials)...

Best trial:
  Val loss: 0.0408
  Config:
    lr: 0.0048
    weight_decay: 0.0113
    hidden_size: 64
    n_layers: 2
    dropout_p: 0.3636
    activation_name: relu

Hyperparameter importance (higher = more impact on loss):
  lr                  : 0.653  ███████████████████
  weight_decay        : 0.227  ██████
  hidden_size         : 0.043  █
  n_layers            : 0.036  █
  activation_name     : 0.023  
  dropout_p           : 0.018  

Top 10 trials (sorted by val_loss):
  Trial   Val Loss         LR         WD   Hidden
--------------------------------------------------
     49     0.0408   4.76e-03   1.13e-02       64
     25     0.0503   1.84e-02   1.35e-03       64
     43     0.0516   2.66e-03   9.85e-04       64
     37     0.0519   3.91e-03   3.66e-04       64
     34     0.0553   8.11e-03   1.35e-03       64
     44     0.0560   2.99e-03   9.08e-04       64
     48     0.0578   3.20e-03   1.69e-03      128
     33     0.0580   8.86

In [9]:
# TensorBoard
# !pip install tensorboard

from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter(log_dir = 'runs/experiment_01')

In [10]:
# Logging Scalars (Loss, Accuaracy, LR)
from torch.utils.tensorboard import SummaryWriter
import torch.nn as nn
import torch.optim as optim

writer = SummaryWriter('run/experiment_02')

for epoch in range(n_epochs):
    train_loss, train_acc = run_train_epoch(...)
    val_loss, val_acc = run_val_epoch(...)
    
    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar('Loss / train', train_loss, epoch)
    writer.add_scalar('Loss / val', val_loss, epoch)
    writer.add_scalar('Accuracy / train', train_acc, epoch)
    writer.add_scalar('Accuracy / val', val_acc, epoch)
    writer.add_scalar('LR', current_lr, epoch)

writer.close()

# Losgging Weight Histograms
for epoch in range(n_epochs):
    # training
    # log histograms every 5 epochs
    if (epoch + 1) % 5 == 0:
        for name, param in model.named_parameters():
            writer.add_histogram(f'weights / {name}', param.data, epoch)
            if param.grid is not None:
                writer.add_histogram(f'grads / {name}'. param.grad, epoch)


# Logging Gradient Flow
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

def plot_gradient_flow(model) -> plt.Figure:
    mean_grads, max_grads, layers = [], [], []
    for name, param in model.named_parameters():
        if param.grid is not None and 'bias' not in name:
            mean_grads.append(param.grad.abs().mean().item())
            max_grads.append(param.grad.abs().max().item())
            layers.append(name.replace('.weight', ''))
    fig, ax = plt.subplots(figsize=(max(8, len(layers) * 1.2), 4))
    x = np.arange(len(layers))
    ax.bar(x - 0.2, mean_grads, 0.4, label='Mean |grad|', color='steelblue', alpha=0.8)
    ax.bar(x + 0.2, max_grads,  0.4, label='Max |grad|',  color='salmon',    alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(layers, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Gradient magnitude')
    ax.set_title('Gradient Flow')
    ax.legend()
    ax.set_yscale('log')   # log scale makes vanishing gradients visible
    ax.axhline(y=1e-6, color='red', linestyle='--', alpha=0.5,
               label='Vanishing threshold')
    fig.tight_layout()
    return fig
    
for epoch in range(n_epochs):
    model.train()
    for bX, by in train_loader:
        bX, by = bX.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bX), by)
        loss.backward()
        # Log gradient flow after backward but before optimizer step
        break   # just use first batch for gradient flow logging

    if (epoch + 1) % 10 == 0:
        fig = plot_gradient_flow(model)
        writer.add_figure('gradient_flow', fig, epoch)
        plt.close(fig)
        # rest training program

NameError: name 'n_epochs' is not defined

In [11]:
# Comparing multiple runs in TensorBoard
writer_a = SummaryWriter('runs/lr_le-3')
writer_b = SummaryWriter('runs/lr_le-4')
# TensorBoard automatically shows both on the same plot
# Navigate to http://localhost:6006, select both runs

In [12]:
# Expected Initial Loss (Sanity Check)
import torch
import math

def expected_initial_loss(task, n_classes = None):
    if task == 'binary':
        return -math.log(0.5)
    elif task == 'multiclass':
        return math.log(n_classes)
    elif task == 'regression':
        return '~ variance of y_train (depends on our data)'

print("Expected initial losses for random model:")
print(f"  Binary classification:     {expected_initial_loss('binary'):.4f}")
print(f"  4-class classification:    {expected_initial_loss('multiclass', 4):.4f}")
print(f"  10-class classification:   {expected_initial_loss('multiclass', 10):.4f}")
print(f"  Regression:                {expected_initial_loss('regression')}")

# In practice: train for 1 step and check
model = build_model(hidden_size=64, dropout_p=0.0)
criterion = nn.BCEWithLogitsLoss()

with torch.no_grad():
    X_check = X_tr[:64].to(device)
    y_check = y_tr[:64].to(device)
    initial_loss = criterion(model(X_check), y_check)

print(f"\nActual initial loss:  {initial_loss.item():.4f}")
print(f"Expected (~BCE for p=0.5): 0.6931")
print(f"Match? {'✅ Yes' if abs(initial_loss.item() - 0.693) < 0.2 else '❌ No — check your model!'}")

Expected initial losses for random model:
  Binary classification:     0.6931
  4-class classification:    1.3863
  10-class classification:   2.3026
  Regression:                ~ variance of y_train (depends on our data)

Actual initial loss:  0.6799
Expected (~BCE for p=0.5): 0.6931
Match? ✅ Yes


In [13]:
# Tiny Batch Overfit Test
def tiny_batch_overfit_test(model, criterion, optimizer, n_samples = 5, n_epochs = 200, device = 'cpu'):
    X_tiny = X_tr[:n_samples].to(device)
    y_tiny = y_tr[:n_samples].to(device)

    print(f"Tiny batch overfit test: {n_samples} samples, {n_epochs} epochs")
    print(f"Input shape:  {X_tiny.shape}")
    print(f"Label shape:  {y_tiny.shape}")
    print(f"Expected: loss should reach < 0.01 (near perfect memorization)\n")

    model.train()
    losses = []

    for epoch in range(n_epochs):
        optimizer.zero_grad()
        output = model(X_tiny)
        loss   = criterion(output, y_tiny)

        if torch.isnan(loss):
            print(f"  ❌ NaN loss at epoch {epoch}!")
            print(f"     Output values: {output.detach()}")
            break

        loss.backward()
        optimizer.step()
        losses.append(loss.item())

        if (epoch + 1) % 50 == 0:
            print(f"  Epoch {epoch+1:>4}: loss = {loss.item():.6f}")

    final_loss = losses[-1] if losses else float('inf')
    success    = final_loss < 0.01

    print(f"\n  Final loss: {final_loss:.6f}")
    print(f"  Result: {'✅ PASS — model can overfit (code is correct)' if success else '❌ FAIL — model cannot overfit (check implementation!)'}")
    return success

test_model = build_model(hidden_size=128, dropout_p=0.0)
test_opt   = optim.AdamW(test_model.parameters(), lr=0.01)
passed     = tiny_batch_overfit_test(test_model, nn.BCEWithLogitsLoss(),
                                      test_opt, n_samples=5, device=device)

Tiny batch overfit test: 5 samples, 200 epochs
Input shape:  torch.Size([5, 15])
Label shape:  torch.Size([5, 1])
Expected: loss should reach < 0.01 (near perfect memorization)

  Epoch   50: loss = 0.000000
  Epoch  100: loss = 0.000000
  Epoch  150: loss = 0.000000
  Epoch  200: loss = 0.000000

  Final loss: 0.000000
  Result: ✅ PASS — model can overfit (code is correct)


In [14]:
# NaN Loss Debugging
def debug_nan_loss(model, criterion, X_batch, y_batch, device='cpu'):
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)
    model.eval()

    print("=== NaN Loss Diagnostics ===\n")

    # 1. Check input data
    print(f"1. Input data:")
    print(f"   NaN in X: {torch.isnan(X_batch).any().item()}")
    print(f"   Inf in X: {torch.isinf(X_batch).any().item()}")
    print(f"   X range:  [{X_batch.min():.3f}, {X_batch.max():.3f}]")
    print(f"   NaN in y: {torch.isnan(y_batch).any().item()}")
    print(f"   y range:  [{y_batch.min():.3f}, {y_batch.max():.3f}]")

    # 2. Trace through each layer
    print(f"\n2. Layer-by-layer activation check:")
    x = X_batch
    for i, layer in enumerate(model):
        with torch.no_grad():
            x = layer(x)
        has_nan = torch.isnan(x).any().item()
        has_inf = torch.isinf(x).any().item()
        x_min, x_max = x.min().item(), x.max().item()
        status = "❌ NaN!" if has_nan else ("❌ Inf!" if has_inf else "✅ OK")
        print(f"   Layer {i} ({layer.__class__.__name__}): "
              f"range=[{x_min:.3f}, {x_max:.3f}]  {status}")
        if has_nan or has_inf:
            print(f"   ↑ First occurrence of NaN/Inf. Problem is in or before this layer.")
            break

    # 3. Check gradients (if backward was already called)
    print(f"\n3. Gradient check:")
    for name, param in model.named_parameters():
        if param.grad is not None:
            has_nan = torch.isnan(param.grad).any().item()
            if has_nan:
                print(f"   ❌ NaN gradient in {name}!")
        else:
            print(f"   ⚠  No gradient for {name} — was backward() called?")

    # 4. Recommendations
    print(f"\n4. Likely causes and fixes:")
    print(f"   - Learning rate too large → try 10× smaller")
    print(f"   - Exploding gradients     → add clip_grad_norm_(model.parameters(), 1.0)")
    print(f"   - NaN in data             → run preprocessing again")
    print(f"   - Log(0) in loss          → use BCEWithLogitsLoss not BCELoss+log")

In [15]:
def analyze_gradient_flow(model, verbose=True) -> dict:
    stats = {}
    for name, param in model.named_parameters():
        if param.grad is None:
            continue

        grad = param.grad.abs()
        stats[name] = {'mean':  grad.mean().item(), 'max':   grad.max().item(),
            'min':   grad.min().item(), 'std':   grad.std().item()}

    if verbose:
        print(f"\n{'Layer':<35} {'Mean Grad':>12} {'Max Grad':>12} {'Status':>12}")
        for name, s in stats.items():
            if s['mean'] < 1e-7:
                status = "⚠  VANISHING"
            elif s['max'] > 100:
                status = "⚠  EXPLODING"
            else:
                status = "✅ OK"
            print(f"{name:<35} {s['mean']:>12.2e} {s['max']:>12.2e} {status:>12}")

    return stats

# ─── Integration in training loop ───
for epoch in range(n_epochs):
    model.train()
    for batch_idx, (bX, by) in enumerate(train_loader):
        bX, by = bX.to(device), by.to(device)

        optimizer.zero_grad()
        output = model(bX)
        loss   = criterion(output, by)
        loss.backward()

        # Analyze gradients on first batch of first few epochs
        if epoch < 3 and batch_idx == 0:
            print(f"\n--- Gradient Analysis (Epoch {epoch+1}) ---")
            analyze_gradient_flow(model, verbose=True)

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

NameError: name 'n_epochs' is not defined

In [16]:
# Minimal Reproducibility Setup
import torch
import numpy as np
import random
import os
import json
from datetime import datetime

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Makes convolutions deterministic (slower but reproducible)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

    os.environ['PYTHONHASHSEED'] = str(seed)

def save_experiment(config: dict, metrics: dict,
                     model: nn.Module, run_name: str = None):
    if run_name is None:
        run_name = datetime.now().strftime('%Y%m%d_%H%M%S')

    os.makedirs(f'experiments/{run_name}', exist_ok=True)

    # Save config and metrics as JSON
    experiment_data = {
        'run_name':    run_name,
        'timestamp':   datetime.now().isoformat(),
        'config':      config,
        'metrics':     metrics,
        'torch_version': torch.__version__,
        'cuda_available': torch.cuda.is_available(),
    }
    with open(f'experiments/{run_name}/experiment.json', 'w') as f:
        json.dump(experiment_data, f, indent=2, default=str)

    # Save model weights
    torch.save(model.state_dict(),
                f'experiments/{run_name}/model.pth')
    print(f"Experiment saved to: experiments/{run_name}/")
    
    return run_name


# ─── Usage ───
set_seed(42)   # call this BEFORE building model or creating DataLoader

config = {
    'lr':           0.001,
    'weight_decay': 0.01,
    'hidden_size':  128,
    'dropout_p':    0.3,
    'n_epochs':     50,
    'batch_size':   64,
    'optimizer':    'AdamW',
    'scheduler':    'CosineAnnealingLR',
}

# ... build model, train ...

metrics = {
    'val_loss':     0.3241,
    'val_accuracy': 87.3,
    'val_f1':       0.862,
    'best_epoch':   38,
}

# model = ... (your trained model)
# save_experiment(config, metrics, model, run_name='churn_v1_baseline')

In [17]:
# Complete Checkpoint Pattern
def save_checkpoint(path: str, epoch: int, model: nn.Module, optimizer: optim.Optimizer,
                     scheduler, val_loss: float, config: dict):
    torch.save({
        'epoch':            epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'val_loss':         val_loss,
        'config':           config,
    }, path)


def load_checkpoint(path: str, model: nn.Module, optimizer: optim.Optimizer = None, scheduler = None):
    checkpoint = torch.load(path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if scheduler is not None and checkpoint['scheduler_state_dict'] is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    start_epoch = checkpoint['epoch'] + 1
    val_loss    = checkpoint['val_loss']

    print(f"Resumed from checkpoint: epoch {checkpoint['epoch']}, "
          f"val_loss={val_loss:.4f}")

    return start_epoch, val_loss


# ─── Usage in training loop ───
best_val_loss = float('inf')
for epoch in range(n_epochs):
    # ... training and validation ...

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            path='best_checkpoint.pth',
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            val_loss=val_loss,
            config=config
        )

    # Also save periodic checkpoints (safety net against crashes)
    if (epoch + 1) % 10 == 0:
        save_checkpoint(
            path=f'checkpoint_epoch_{epoch+1}.pth',
            epoch=epoch, model=model,
            optimizer=optimizer, scheduler=scheduler,
            val_loss=val_loss, config=config
        )

NameError: name 'n_epochs' is not defined

In [18]:
# Complete Script
"""
Production training script template.
Integrates: seed setting, config tracking, TensorBoard logging,
            early stopping, gradient analysis, and checkpointing.
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import TensorDataset, DataLoader
import json, os
from datetime import datetime

# 1. CONFIG — all hyperparameters in one place
config = {
    # Data
    'n_features' : 20, 'batch_size' : 64, 'seed' : 42,
    # Architecture
    'hidden_sizes' : [128, 64, 32], 'dropout_p': 0.3, 'use_batchnorm' : True,
    # Training
    'n_epochs' : 60, 'lr': 1e-3, 'weight_decay': 0.01, 'warmup_epochs': 5, 'max_grad_norm': 1.0,
    # Early stopping
    'patience' : 10, 'min_delta': 1e-4,
    # Logging 
    'run_name': f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
}

# 2. SETUP
set_seed(config['seed'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(f"runs/{config['run_name']}", exist_ok=True)

writer = SummaryWriter(f"runs/{config['run_name']}")

# Save config immediately
with open(f"runs/{config['run_name']}/config.json", 'w') as f:
    json.dump(config, f, indent=2)

# 3. DATA
torch.manual_seed(config['seed'])
n = 2000
X = torch.randn(n, config['n_features'])
y = (X[:, :4].sum(1) > 0).long()

split = int(0.8 * n)
X_tr, X_val = X[:split], X[split:]
y_tr, y_val = y[:split], y[split:]

train_loader = DataLoader(TensorDataset(X_tr, y_tr),
                           batch_size=config['batch_size'], shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val),
                           batch_size=256, shuffle=False)

# 4. MODEL
def build_production_model(in_features, hidden_sizes, dropout_p, use_batchnorm):
    layers, prev = [], in_features
    for h in hidden_sizes:
        layers += [nn.Linear(prev, h), nn.ReLU()]
        if use_batchnorm: layers.append(nn.BatchNorm1d(h))
        if dropout_p > 0: layers.append(nn.Dropout(dropout_p))
        prev = h
    layers.append(nn.Linear(prev, 2))   # 2-class output
    model = nn.Sequential(*layers)
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)
    return model

model = build_production_model(
    config['n_features'],
    config['hidden_sizes'],
    config['dropout_p'],
    config['use_batchnorm']
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
writer.add_text('model/n_params', str(n_params))

# 5. TINY BATCH TEST — run before full training
test_model = build_production_model(
    config['n_features'], config['hidden_sizes'],
    0.0, False   # no regularization for the test
).to(device)
test_opt = optim.Adam(test_model.parameters(), lr=0.01)
tiny_batch_overfit_test(test_model, nn.CrossEntropyLoss(),
                         test_opt, n_samples=4, device=device)

# 6. OPTIMIZER + SCHEDULER
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(),
                         lr=config['lr'],
                         weight_decay=config['weight_decay'])

scheduler = CosineAnnealingLR(optimizer,
                                T_max=config['n_epochs'],
                                eta_min=1e-6)

# 7. TRAINING LOOP
best_val_loss  = float('inf')
patience_count = 0
history        = []

for epoch in range(config['n_epochs']):

    # ── Train ──
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0

    for bX, by in train_loader:
        bX, by = bX.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bX)
        loss   = criterion(logits, by)
        loss.backward()

        # Gradient clipping
        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(), config['max_grad_norm']
        )

        optimizer.step()
        t_loss    += loss.item() * bX.size(0)
        t_correct += (logits.argmax(1) == by).sum().item()
        t_total   += bX.size(0)

    # ── Validate ──
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for bX, by in val_loader:
            bX, by = bX.to(device), by.to(device)
            logits  = model(bX)
            loss    = criterion(logits, by)
            v_loss    += loss.item() * bX.size(0)
            v_correct += (logits.argmax(1) == by).sum().item()
            v_total   += bX.size(0)

    # ── Metrics ──
    tl = t_loss / t_total
    vl = v_loss / v_total
    ta = t_correct / t_total * 100
    va = v_correct / v_total * 100
    lr = optimizer.param_groups[0]['lr']

    # ── TensorBoard logging ──
    writer.add_scalar('Loss/train',     tl, epoch)
    writer.add_scalar('Loss/val',       vl, epoch)
    writer.add_scalar('Accuracy/train', ta, epoch)
    writer.add_scalar('Accuracy/val',   va, epoch)
    writer.add_scalar('LR',             lr, epoch)

    if (epoch + 1) % 10 == 0:
        for name, param in model.named_parameters():
            writer.add_histogram(f'weights/{name}', param.data, epoch)
            if param.grad is not None:
                writer.add_histogram(f'grads/{name}', param.grad, epoch)

    # ── Scheduler ──
    scheduler.step()

    # ── Early stopping + checkpointing ──
    if vl < best_val_loss - config['min_delta']:
        best_val_loss  = vl
        patience_count = 0
        save_checkpoint('best_model.pth', epoch, model,
                         optimizer, scheduler, vl, config)
    else:
        patience_count += 1

    history.append({'epoch': epoch+1, 'tl': tl, 'vl': vl, 'ta': ta, 'va': va})

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:>3} | LR: {lr:.2e} | "
              f"Train: {tl:.4f} ({ta:.1f}%) | "
              f"Val: {vl:.4f} ({va:.1f}%) | "
              f"Patience: {patience_count}/{config['patience']}")

    if patience_count >= config['patience']:
        print(f"\n⛔ Early stopping at epoch {epoch+1}")
        break

# 8. FINAL EVALUATION AND SAVE
start_epoch, _ = load_checkpoint('best_model.pth', model)
writer.add_text('results/best_val_loss', f"{best_val_loss:.4f}")

final_metrics = {
    'best_val_loss': best_val_loss,
    'total_epochs':  len(history),
}
save_experiment(config, final_metrics, model, config['run_name'])
writer.close()

print(f"\n✅ Training complete.")
print(f"   Best val loss: {best_val_loss:.4f}")
print(f"   Epochs run:    {len(history)}")

Model parameters: 13,538
Tiny batch overfit test: 4 samples, 200 epochs
Input shape:  torch.Size([4, 20])
Label shape:  torch.Size([4])
Expected: loss should reach < 0.01 (near perfect memorization)

  Epoch   50: loss = 0.000000
  Epoch  100: loss = 0.000000
  Epoch  150: loss = 0.000000
  Epoch  200: loss = 0.000000

  Final loss: 0.000000
  Result: ✅ PASS — model can overfit (code is correct)
Epoch  10 | LR: 9.46e-04 | Train: 0.2147 (91.3%) | Val: 0.1682 (93.2%) | Patience: 0/10
Epoch  20 | LR: 7.73e-04 | Train: 0.1617 (92.9%) | Val: 0.1351 (94.5%) | Patience: 0/10
Epoch  30 | LR: 5.27e-04 | Train: 0.1475 (94.0%) | Val: 0.1095 (94.8%) | Patience: 0/10
Epoch  40 | LR: 2.74e-04 | Train: 0.1223 (95.4%) | Val: 0.1084 (95.8%) | Patience: 1/10
Epoch  50 | LR: 8.16e-05 | Train: 0.0993 (95.6%) | Val: 0.0988 (95.5%) | Patience: 0/10
Epoch  60 | LR: 1.68e-06 | Train: 0.1166 (95.2%) | Val: 0.1042 (95.8%) | Patience: 4/10
Resumed from checkpoint: epoch 55, val_loss=0.0983
Experiment saved to: e

In [19]:
# Mini Exercise: Complete Reproducible experiment that tunes hyperparameters with optuna and 
# logs results with TensorBoard

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import TensorDataset, DataLoader
import optuna
import json, os, random
import numpy as np
from datetime import datetime

optuna.logging.set_verbosity(optuna.logging.WARNING)

# SETUP
SEED   = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

set_seed(SEED)

# Dataset
n = 2500
X = torch.randn(n, 20)
y = (X[:, :3].sum(1) - X[:, 3:5].sum(1) > 0).long()

X_tr, X_val = X[:2000], X[2000:]
y_tr, y_val = y[:2000], y[2000:]

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)


# MODEL
def build_from_config(cfg: dict) -> nn.Module:
    layers, prev = [], 20
    for h in cfg['hidden_sizes']:
        layers += [nn.Linear(prev, h), nn.ReLU()]
        if cfg['use_bn']:     layers.append(nn.BatchNorm1d(h))
        if cfg['dropout'] > 0: layers.append(nn.Dropout(cfg['dropout']))
        prev = h
    layers.append(nn.Linear(prev, 2))
    model = nn.Sequential(*layers).to(DEVICE)
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)
    return model


# OPTUNA OBJECTIVE
def objective(trial: optuna.Trial) -> float:
    set_seed(SEED + trial.number)   # reproducible per trial

    # Suggest config
    n_layers = trial.suggest_int('n_layers', 1, 4)
    width    = trial.suggest_categorical('width', [64, 128, 256])
    cfg = {
        'hidden_sizes': [width] * n_layers,
        'dropout':      trial.suggest_float('dropout', 0.0, 0.5),
        'use_bn':       trial.suggest_categorical('use_bn', [True, False]),
        'lr':           trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-1, log=True),
    }

    model     = build_from_config(cfg)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(),
                             lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-6)

    best_val_loss = float('inf')

    for epoch in range(40):
        model.train()
        for bX, by in train_loader:
            bX, by = bX.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(bX), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            vl = criterion(model(X_val.to(DEVICE)), y_val.to(DEVICE)).item()

        best_val_loss = min(best_val_loss, vl)
        trial.report(vl, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_loss


# RUN STUDY
print("Running Optuna hyperparameter search (40 trials)...")

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
)
study.optimize(objective, n_trials=40)

print(f"\n✅ Best val loss: {study.best_value:.4f}")
print(f"   Config: {study.best_params}")


# RETRAIN BEST CONFIG WITH FULL LOGGING
print("\nRetraining best configuration with full logging...")

set_seed(SEED)
best_cfg = {
    'hidden_sizes': [study.best_params['width']] * study.best_params['n_layers'],
    'dropout':      study.best_params['dropout'],
    'use_bn':       study.best_params['use_bn'],
    'lr':           study.best_params['lr'],
    'weight_decay': study.best_params['weight_decay'],
}

run_name = f"best_run_{datetime.now().strftime('%H%M%S')}"
writer   = SummaryWriter(f'runs/{run_name}')
model    = build_from_config(best_cfg)
opt      = optim.AdamW(model.parameters(),
                        lr=best_cfg['lr'], weight_decay=best_cfg['weight_decay'])
sched    = CosineAnnealingLR(opt, T_max=60, eta_min=1e-6)
crit     = nn.CrossEntropyLoss()

best_val, patience = float('inf'), 0

for epoch in range(60):
    model.train()
    tl, tc, tt = 0.0, 0, 0
    for bX, by in train_loader:
        bX, by = bX.to(DEVICE), by.to(DEVICE)
        opt.zero_grad()
        logits = model(bX)
        loss   = crit(logits, by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl += loss.item() * bX.size(0)
        tc += (logits.argmax(1) == by).sum().item()
        tt += bX.size(0)
    sched.step()

    model.eval()
    vl, vc, vt = 0.0, 0, 0
    with torch.no_grad():
        for bX, by in val_loader:
            bX, by = bX.to(DEVICE), by.to(DEVICE)
            logits = model(bX)
            vl += crit(logits, by).item() * bX.size(0)
            vc += (logits.argmax(1) == by).sum().item()
            vt += bX.size(0)

    writer.add_scalar('Loss/train',     tl/tt, epoch)
    writer.add_scalar('Loss/val',       vl/vt, epoch)
    writer.add_scalar('Accuracy/train', tc/tt*100, epoch)
    writer.add_scalar('Accuracy/val',   vc/vt*100, epoch)
    writer.add_scalar('LR', opt.param_groups[0]['lr'], epoch)

    if vl/vt < best_val - 1e-4:
        best_val, patience = vl/vt, 0
        torch.save(model.state_dict(), f'runs/{run_name}/best_model.pth')
    else:
        patience += 1
    if patience >= 10:
        print(f"  Early stopping at epoch {epoch+1}")
        break

    if (epoch + 1) % 15 == 0:
        print(f"  Epoch {epoch+1}: val_loss={vl/vt:.4f}, val_acc={vc/vt*100:.1f}%")

writer.close()
print(f"\nFinal best val loss: {best_val:.4f}")
print(f"Run logged to: runs/{run_name}")
print(f"Launch TensorBoard: tensorboard --logdir=runs")

Running Optuna hyperparameter search (40 trials)...

✅ Best val loss: 0.0275
   Config: {'n_layers': 2, 'width': 128, 'dropout': 0.14607232426760908, 'use_bn': False, 'lr': 0.0037183641805732083, 'weight_decay': 6.290644294586152e-05}

Retraining best configuration with full logging...
  Epoch 15: val_loss=0.0927, val_acc=96.4%
  Epoch 30: val_loss=0.0670, val_acc=96.6%
  Early stopping at epoch 31

Final best val loss: 0.0580
Run logged to: runs/best_run_000933
Launch TensorBoard: tensorboard --logdir=runs
